In [173]:
import pandas as pd
import numpy as np

In [237]:
bein_original = pd.read_csv(r"S:\22.07.26\30596407_BEINDATANEWRPT.CSV", dtype='str')
swap_original = pd.read_csv(r"S:\Sheikh\30592658_SWAPRPT.CSV",dtype='str')


In [ ]:

swap_original['Swap Datetime'] = pd.to_datetime(
    swap_original['Swap Date'].str.split().str[0] + ' ' + swap_original['Swap Time'],
    format='%d/%m/%Y %I:%M:%S %p'
)
swap_original = swap_original.loc[swap_original['Item']!='ART Smart Card']

swap = swap_original[['Swap Datetime','Subscriber Number', 'Subscriber Type',
       'Replacement Number', 'Item', 'Old Serial Number', 'New Serial Number',
       ]]

exclude_types = ['Clubs','Muds','Hotels','Clubs Free',"In House Demo's",'Dealers demo','VIP_CNE','Bein Companies PV', 'Temp','Temp OSN']

swap = swap.loc[~swap['Subscriber Type'].isin(exclude_types)]
swap = swap.sort_values(['Subscriber Number','Swap Datetime'], ascending=[True,True])
swap_sc = swap.loc[swap['Item']=='Smartcard']
swap_dec = swap.loc[swap['Item']=='Decoder']


swap_sc=swap_sc.rename(columns={'Old Serial Number':'Old Smartcard Number','New Serial Number':'New Smartcard Number' })
swap_sc['Old Decoder Number']=pd.NA
swap_sc['New Decoder Number']=pd.NA

swap_dec=swap_dec.rename(columns={'Old Serial Number':'Old Decoder Number','New Serial Number':'New Decoder Number' })
swap_dec['Old Smartcard Number']=pd.NA
swap_dec['New Smartcard Number']=pd.NA

all = pd.concat([swap_dec,swap_sc])
all = all.sort_values(['Subscriber Number','Swap Datetime'],ascending=[True,True])


In [188]:
bein_original.columns

Index(['Customer Number', 'Customer Type', 'Entity', 'Contract Number',
       'Start Date', 'End Date', 'Plan', 'Status', 'Decoder',
       'Item Description STB', 'Smart Card', 'Item Description SC',
       'Next Billing Date', 'Billing Cycle', 'PPV Balance', 'Customer Balance',
       'Outstanding Balance'],
      dtype='object')

In [216]:
bein_original.loc[bein_original['Customer Number']=='9852638']

bein_base = bein_original[['Customer Number', 'Customer Type',  'Decoder',
        'Smart Card',
       ]].drop_duplicates(subset='Customer Number')
bein_base = bein_base.loc[~bein_base['Customer Type'].isin(exclude_types)]

bein_base['Swap Datetime'] = pd.to_datetime('2018-01-01')
bein_base= bein_base.rename(columns={'Customer Number':'Subscriber Number','Customer Type':'Subscriber Type','Decoder':'New Decoder Number','Smart Card':'New Smartcard Number'})

bein_base

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime
0,9999654,beIN Quartar Installment,0303293316,42916623426,2018-01-01
3,9999475,beIN Bi Installment,0303293326,42916729918,2018-01-01
16,9998056,BeIN sports CC,0302788561,42768375273,2018-01-01
17,9997853,CNE Subscriber,0303319270,42916761929,2018-01-01
20,9997804,beIN Quartar Installment,0319705313,10388802489,2018-01-01
...,...,...,...,...,...
3038960,10005578,CNE Subscriber,0319593070,42916623509,2018-01-01
3038963,10005288,CNE Subscriber,0296574830,42912373976,2018-01-01
3038971,1000212,CNE Subscriber,0278680677,42916939681,2018-01-01
3038972,10000224,CNE Subscriber,0302778636,42916818737,2018-01-01


In [217]:
swap_dec.loc[swap_dec['Subscriber Number']== '10038709']

,Swap Datetime,Subscriber Number,Subscriber Type,Replacement Number,Item,Old Decoder Number,New Decoder Number,Old Smartcard Number,New Smartcard Number
15810,2019-06-01 14:33:16,10038709,CNE Subscriber,552,Decoder,0303337309,0349823962,<NA>,<NA>


In [218]:
base_sc = swap_sc.sort_values(['Subscriber Number', 'Swap Datetime']).drop_duplicates(subset=['Subscriber Number'], keep='first')
base_sc['Swap Datetime'] = pd.to_datetime('2018-01-01')
base_sc = base_sc.drop(columns=['Replacement Number','Item','Old Decoder Number','New Smartcard Number']).rename(columns={'Old Smartcard Number':'New Smartcard Number'})


base_sc.loc[base_sc['Subscriber Number']== '10038709']


base_dec = swap_dec.sort_values(['Subscriber Number', 'Swap Datetime']).drop_duplicates(subset=['Subscriber Number'], keep='first')
base_dec['Swap Datetime'] = pd.to_datetime('2018-01-01')
base_dec = base_dec.drop(columns=['Replacement Number','Item','New Decoder Number','Old Smartcard Number']).rename(columns={'Old Decoder Number':'New Decoder Number'})


base_dec.loc[base_dec['Subscriber Number']== '10038709']



,Swap Datetime,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number
15810,2018-01-01,10038709,CNE Subscriber,0303337309,<NA>


In [219]:
cols = [
    'Old Decoder Number',
    'New Decoder Number',
    'Old Smartcard Number',
    'New Smartcard Number'
]

history = all.sort_values(
    ['Subscriber Number', 'Swap Datetime']
).copy()

history[cols] = (
    history.groupby('Subscriber Number', sort=False)[cols]
           .transform(lambda x: x.ffill())
)


history = history.drop(columns=['Replacement Number','Item','Old Smartcard Number','Old Decoder Number']).drop_duplicates()
# history = history.loc[~history['Subscriber Type'].isin(exclude_types)]


C:\Users\mturky\AppData\Local\Temp\1\ipykernel_25608\3109962901.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.ffill())


In [220]:
print(history.columns)
print(bein_base.columns)

Index(['Swap Datetime', 'Subscriber Number', 'Subscriber Type',
       'New Decoder Number', 'New Smartcard Number'],
      dtype='object')
Index(['Subscriber Number', 'Subscriber Type', 'New Decoder Number',
       'New Smartcard Number', 'Swap Datetime'],
      dtype='object')


In [221]:
history = (
    history
    .drop_duplicates(subset=['Subscriber Number','Swap Datetime','New Decoder Number'],keep='last')
    .drop_duplicates(subset=['Subscriber Number','Swap Datetime','New Smartcard Number'],keep='last')
)


history = pd.concat([base_sc,history])
history.sort_values(['Subscriber Number','Swap Datetime'])
history["New Smartcard Number"] = (history.groupby("Subscriber Number")["New Smartcard Number"].ffill())
history = history.loc[history['Swap Datetime']!='2018-01-01']



history = pd.concat([base_dec,history])
history.sort_values(['Subscriber Number','Swap Datetime'])
history["New Decoder Number"] = (history.groupby("Subscriber Number")["New Decoder Number"].ffill())
history = history.loc[history['Swap Datetime']!='2018-01-01']



history = pd.concat([bein_base, history])
history.sort_values(['Subscriber Number','Swap Datetime'])
history["New Decoder Number"] = (history.groupby("Subscriber Number")["New Decoder Number"].ffill())
history["New Smartcard Number"] = (history.groupby("Subscriber Number")["New Smartcard Number"].ffill())
history = history.loc[history['Swap Datetime']!='2018-01-01']




In [224]:
bein_base = bein_base.loc[~bein_base['Subscriber Number'].isin(history['Subscriber Number'])]
bein_base

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime
0,9999654,beIN Quartar Installment,0303293316,42916623426,2018-01-01
3,9999475,beIN Bi Installment,0303293326,42916729918,2018-01-01
16,9998056,BeIN sports CC,0302788561,42768375273,2018-01-01
17,9997853,CNE Subscriber,0303319270,42916761929,2018-01-01
20,9997804,beIN Quartar Installment,0319705313,10388802489,2018-01-01
...,...,...,...,...,...
3038960,10005578,CNE Subscriber,0319593070,42916623509,2018-01-01
3038963,10005288,CNE Subscriber,0296574830,42912373976,2018-01-01
3038971,1000212,CNE Subscriber,0278680677,42916939681,2018-01-01
3038972,10000224,CNE Subscriber,0302778636,42916818737,2018-01-01


In [225]:
history = pd.concat([bein_base,history])

In [236]:
history.loc[history['Subscriber Number']=='10709659']



,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime


In [ ]:
t = history.groupby(['Subscriber Number','Subscriber Type']).agg(count=('Subscriber Number','count')).reset_index()
t.loc[t['count']>3].head(30)

,Subscriber Number,Subscriber Type,count
372,12214167,CNE Subscriber,4
403,12290430,beIN Bi Installment,4
2075,13781643,beIN Quartar Installment,4
2390,13841175,beIN Quartar Installment,4
2458,13856581,beIN Quartar Installment,5
2827,13917027,CNE Subscriber,4
3182,13979101,beIN Quartar Installment,5
3258,13991461,Illegal Network,4
3664,14055479,Head End,5
3675,14056821,Illegal Network,4


In [234]:
history.loc[history['Subscriber Number']=='14080580'].sort_values('Swap Datetime')

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime
25693,14080580,beIN Quartar Installment,0349864106,10679685635,2020-11-12 16:58:35
58047,14080580,beIN Quartar Installment,0349668187,10679685635,2021-07-12 11:48:12
27991,14080580,beIN Quartar Installment,0351939234,10709034598,2022-07-21 14:44:15
20699,14080580,beIN Quartar Installment,0351939234,10719502238,2023-02-02 12:20:45


In [ ]:
history.loc[history['New Decoder Number'].isna()]

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime


In [227]:
history.loc[history['New Decoder Number'].isna()]

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime
9391,658634,CNE Subscriber,NaN,NaN,2018-01-01
60989,19206718,beIN Bi Installment,NaN,10732850648,2018-01-01
136412,19165304,CNE Subscriber,NaN,NaN,2018-01-01
136538,19165243,beIN Quartar Installment,NaN,10732747133,2018-01-01
277490,19112648,beIN Quartar Installment,NaN,NaN,2018-01-01
...,...,...,...,...,...
1477968,17847315,CNE Subscriber,NaN,NaN,2018-01-01
1717047,17215594,Bulk DTH customer,NaN,NaN,2018-01-01
1991016,16393910,MCE staff (CNE staff),NaN,NaN,2018-01-01
2747230,14117751,Bulk DTH customer,NaN,NaN,2018-01-01


In [228]:
history.loc[history['New Smartcard Number'].isna()]

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime
9391,658634,CNE Subscriber,NaN,NaN,2018-01-01
24130,19242340,CNE Subscriber,0381382071,NaN,2018-01-01
30860,19235446,beIN Quartar Installment,0315433069,NaN,2018-01-01
31002,19235292,CNE Subscriber,0381362692,NaN,2018-01-01
31613,19234716,beIN Quartar Installment,0381362908,NaN,2018-01-01
...,...,...,...,...,...
1477968,17847315,CNE Subscriber,NaN,NaN,2018-01-01
1717047,17215594,Bulk DTH customer,NaN,NaN,2018-01-01
1991016,16393910,MCE staff (CNE staff),NaN,NaN,2018-01-01
2747230,14117751,Bulk DTH customer,NaN,NaN,2018-01-01


In [ ]:
history

,Subscriber Number,Subscriber Type,New Decoder Number,New Smartcard Number,Swap Datetime
44423,10006697,BeIN sports CC,0349904394,42912458793,2023-01-17 16:15:24
18893,10017195,CNE Subscriber,0349903107,10709096662,2021-09-23 17:03:36
24958,10034845,CNE Subscriber,0360927568,10676369787,2020-03-03 13:47:16
39768,10038709,CNE Subscriber,0303337309,10509031257,2019-06-01 14:32:34
15810,10038709,CNE Subscriber,0349823962,10509031257,2019-06-01 14:33:16
...,...,...,...,...,...
57397,9852638,Illegal Network,0302819763,10691200611,2021-01-24 16:28:25
38657,9919177,CNE Subscriber,0381395162,10738899649,2026-01-06 18:31:09
5707,9970208,beIN Quartar Installment,0302814183,10731089198,2024-06-25 19:52:11
11572,9975465,beIN Bi Installment,0351974423,10708991772,2022-03-02 16:48:38
